# 11 — Model Comparison and Validation  
## Gold Nexus Alpha — Professor-Style Forecasting Platform

**Purpose:** Load completed model artifacts, compare validation/test metrics, rank models, select the official candidate using evidence, and export JSON artifacts for the Model Comparison page.

This notebook is JSON-first and professor-safe:
- It does **not** hardcode a winner.
- It ranks models from available exported metrics.
- It tolerates missing future notebooks/artifacts.
- It avoids failing only because one model artifact filename is slightly different.

**Outputs:**
- `artifacts/validation/validation_summary.json`
- `artifacts/validation/validation_by_model.json`
- `artifacts/validation/validation_by_model.csv`
- `artifacts/validation/model_ranking.json`
- `artifacts/validation/model_ranking.csv`
- `artifacts/validation/residual_diagnostics.json`
- `artifacts/validation/selected_model_summary.json`
- `artifacts/pages/page_model_comparison.json`

## CELL 1 — Repo sync and clean setup

In [ ]:
# =========================
# CELL 1 — REPO SYNC
# =========================

import os
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---- PROJECT SETTINGS ----
# These are set to your current GitHub repo. You can still override them with environment variables if needed.
GITHUB_USERNAME = os.environ.get("GITHUB_USERNAME", "rathee000001")
REPO_NAME = os.environ.get("REPO_NAME", "nyit-gold-intelligence-2026")
BRANCH = os.environ.get("BRANCH", "main")

REPO_URL_PUBLIC = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if IN_COLAB:
    LOCAL_REPO_DIR = Path("/content") / REPO_NAME
else:
    # If you are already inside the repo in VS Code, use the current folder.
    LOCAL_REPO_DIR = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd() / REPO_NAME

def run_cmd(cmd, cwd=None, check=True):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

if IN_COLAB:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        raise ValueError("GITHUB_TOKEN not found in Colab Secrets. Add a secret named GITHUB_TOKEN before running.")
    REPO_URL_AUTH = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
else:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    REPO_URL_AUTH = REPO_URL_PUBLIC

if LOCAL_REPO_DIR.exists() and (LOCAL_REPO_DIR / ".git").exists():
    print(f"Repo exists: {LOCAL_REPO_DIR}")
    run_cmd(["git", "fetch", "origin"], cwd=LOCAL_REPO_DIR)
    run_cmd(["git", "checkout", BRANCH], cwd=LOCAL_REPO_DIR)
    # Rebase keeps local Colab commits cleaner than merge commits.
    pull_result = run_cmd(["git", "pull", "--rebase", "origin", BRANCH], cwd=LOCAL_REPO_DIR, check=False)
    if pull_result.returncode != 0:
        raise RuntimeError(
            "Git pull --rebase failed. Check for local uncommitted changes or conflicts, then rerun. "
            "In Colab this usually means the repo was edited during the session."
        )
else:
    if LOCAL_REPO_DIR.exists() and not (LOCAL_REPO_DIR / ".git").exists():
        print(f"Folder exists but is not a git repo: {LOCAL_REPO_DIR}")
        print("Using a clean clone folder instead.")
        LOCAL_REPO_DIR = LOCAL_REPO_DIR.with_name(LOCAL_REPO_DIR.name + "-clone")
    print(f"Cloning repo into: {LOCAL_REPO_DIR}")
    run_cmd(["git", "clone", "-b", BRANCH, REPO_URL_AUTH, str(LOCAL_REPO_DIR)])

for folder in ["artifacts/validation", "artifacts/pages", "artifacts/models", "artifacts/forecast", "data/aligned"]:
    (LOCAL_REPO_DIR / folder).mkdir(parents=True, exist_ok=True)

print("✅ Repo sync complete.")
print(f"Local repo: {LOCAL_REPO_DIR}")

## CELL 2 — Dependencies

In [ ]:
# =========================
# CELL 2 — DEPENDENCIES
# =========================

import json
import math
import re
import warnings
from pathlib import Path
from datetime import datetime, timezone, date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("✅ Dependencies loaded.")

## CELL 3 — Path setup and flexible artifact detection

In [ ]:
# =========================
# CELL 3 — PATH SETUP + FLEXIBLE INPUT DETECTION
# =========================

REPO = LOCAL_REPO_DIR
MODELS_DIR = REPO / "artifacts/models"

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def first_existing_path(candidates, glob_patterns=None):
    # Return the first existing path from exact candidates.
    # If none exists, try glob patterns.
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p

    glob_patterns = glob_patterns or []
    for pattern in glob_patterns:
        matches = sorted(MODELS_DIR.glob(pattern))
        if matches:
            return matches[0]
    return None

MODEL_ARTIFACT_CANDIDATES = {
    "naive": [
        MODELS_DIR / "naive_results.json",
        MODELS_DIR / "naive_forecast_results.json",
        MODELS_DIR / "naive_model_results.json",
        MODELS_DIR / "baseline_results.json",
        MODELS_DIR / "baseline_model_results.json",
        MODELS_DIR / "naive_moving_average_results.json",
        MODELS_DIR / "baseline_forecast_results.json",
    ],
    "moving_average": [
        MODELS_DIR / "moving_average_results.json",
        MODELS_DIR / "moving_average_forecast_results.json",
        MODELS_DIR / "moving_average_model_results.json",
        MODELS_DIR / "baseline_results.json",
        MODELS_DIR / "baseline_model_results.json",
        MODELS_DIR / "naive_moving_average_results.json",
        MODELS_DIR / "baseline_forecast_results.json",
    ],
    "exponential_smoothing": [
        MODELS_DIR / "exponential_smoothing_results.json",
        MODELS_DIR / "ets_results.json",
        MODELS_DIR / "exp_smoothing_results.json",
    ],
    "regression": [
        MODELS_DIR / "regression_results.json",
        MODELS_DIR / "linear_regression_results.json",
        MODELS_DIR / "multivariate_regression_results.json",
    ],
    "arima": [
        MODELS_DIR / "arima_results.json",
        MODELS_DIR / "arima_model_results.json",
    ],
    "sarimax": [
        MODELS_DIR / "sarimax_results.json",
        MODELS_DIR / "sarimax_model_results.json",
    ],
    "xgboost": [
        MODELS_DIR / "xgboost_results.json",
        MODELS_DIR / "xgboost_candidate_results.json",
        MODELS_DIR / "xgb_results.json",
    ],
    "prophet_optional": [
        MODELS_DIR / "prophet_results.json",
        MODELS_DIR / "prophet_optional_results.json",
    ],
}

MODEL_ARTIFACT_GLOBS = {
    "naive": ["*naive*result*.json", "*baseline*result*.json"],
    "moving_average": ["*moving*average*result*.json", "*baseline*result*.json"],
    "exponential_smoothing": ["*exponential*smoothing*result*.json", "*ets*result*.json"],
    "regression": ["*regression*result*.json"],
    "arima": ["*arima*result*.json"],
    "sarimax": ["*sarimax*result*.json"],
    "xgboost": ["*xgboost*result*.json", "*xgb*result*.json"],
    "prophet_optional": ["*prophet*result*.json"],
}

FORECAST_PATH_CANDIDATES = {
    "naive_moving_average": [
        MODELS_DIR / "baseline_forecast_paths.json",
        MODELS_DIR / "naive_moving_average_forecast_paths.json",
        MODELS_DIR / "baseline_forecast_path.json",
        MODELS_DIR / "naive_moving_average_forecast_path.json",
    ],
    "exponential_smoothing": [
        MODELS_DIR / "exponential_smoothing_forecast_path.json",
        MODELS_DIR / "ets_forecast_path.json",
    ],
    "arima": [
        MODELS_DIR / "arima_forecast_path.json",
    ],
    "sarimax": [
        MODELS_DIR / "sarimax_forecast_path.json",
    ],
    "xgboost": [
        MODELS_DIR / "xgboost_forecast_path.json",
        MODELS_DIR / "xgb_forecast_path.json",
    ],
    "prophet_optional": [
        MODELS_DIR / "prophet_forecast_path.json",
    ],
}

FORECAST_PATH_GLOBS = {
    "naive_moving_average": ["*baseline*forecast*path*.json", "*naive*moving*forecast*path*.json"],
    "exponential_smoothing": ["*exponential*smoothing*forecast*path*.json", "*ets*forecast*path*.json"],
    "arima": ["*arima*forecast*path*.json"],
    "sarimax": ["*sarimax*forecast*path*.json"],
    "xgboost": ["*xgboost*forecast*path*.json", "*xgb*forecast*path*.json"],
    "prophet_optional": ["*prophet*forecast*path*.json"],
}

MODEL_ARTIFACTS = {}
for model_key, candidates in MODEL_ARTIFACT_CANDIDATES.items():
    MODEL_ARTIFACTS[model_key] = first_existing_path(candidates, MODEL_ARTIFACT_GLOBS.get(model_key, []))

FORECAST_PATHS = {}
for path_key, candidates in FORECAST_PATH_CANDIDATES.items():
    FORECAST_PATHS[path_key] = first_existing_path(candidates, FORECAST_PATH_GLOBS.get(path_key, []))

available_artifacts = []
missing_artifacts = []

for model_key, path in MODEL_ARTIFACTS.items():
    if path is not None and path.exists():
        available_artifacts.append({"model_key": model_key, "path": str(path.relative_to(REPO))})
    else:
        missing_artifacts.append({
            "model_key": model_key,
            "path": "not found",
            "expected_examples": [str(p.relative_to(REPO)) for p in MODEL_ARTIFACT_CANDIDATES[model_key][:3]]
        })

available_forecast_paths = []
missing_forecast_paths = []

for path_key, path in FORECAST_PATHS.items():
    if path is not None and path.exists():
        available_forecast_paths.append({"path_key": path_key, "path": str(path.relative_to(REPO))})
    else:
        missing_forecast_paths.append({"path_key": path_key, "path": "not found"})

print("Available model result artifacts:")
display(pd.DataFrame(available_artifacts) if available_artifacts else pd.DataFrame(columns=["model_key", "path"]))

print("Missing model result artifacts:")
display(pd.DataFrame(missing_artifacts) if missing_artifacts else pd.DataFrame(columns=["model_key", "path"]))

print("Available forecast path artifacts:")
display(pd.DataFrame(available_forecast_paths) if available_forecast_paths else pd.DataFrame(columns=["path_key", "path"]))

if len(available_artifacts) == 0:
    raise FileNotFoundError(
        "No model result artifacts found in artifacts/models. "
        "Run at least Notebook 04/05/06/etc. first, or check that artifacts were pushed to GitHub."
    )

## CELL 4 — Model comparison and validation logic

Ranking rule:
1. Prefer **test RMSE** when available.
2. Else use **validation RMSE**.
3. Lower RMSE is better.
4. Ties are broken by MAE, then MAPE.

In [ ]:
# =========================
# CELL 4 — MODEL COMPARISON + VALIDATION
# =========================

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()
OFFICIAL_CUTOFF_DATE = "2026-03-31"

MODEL_DISPLAY_NAMES = {
    "naive": "Naive Forecast",
    "moving_average": "Moving Average",
    "exponential_smoothing": "Exponential Smoothing",
    "regression": "Regression",
    "arima": "ARIMA",
    "sarimax": "SARIMAX",
    "xgboost": "XGBoost Candidate",
    "prophet_optional": "Prophet Optional",
}

MODEL_CATEGORIES = {
    "naive": "baseline",
    "moving_average": "baseline",
    "exponential_smoothing": "classical",
    "regression": "explainable",
    "arima": "classical",
    "sarimax": "economic_time_series",
    "xgboost": "advanced_nonlinear_candidate",
    "prophet_optional": "optional_benchmark",
}

MODEL_SCOPE_TERMS = {
    "naive": [["naive"]],
    "moving_average": [["moving", "average"], ["moving_average"], ["ma"]],
    "exponential_smoothing": [["exponential", "smoothing"], ["ets"]],
    "regression": [["regression"], ["linear"]],
    "arima": [["arima"]],
    "sarimax": [["sarimax"]],
    "xgboost": [["xgboost"], ["xgb"]],
    "prophet_optional": [["prophet"]],
}

def flatten_json(obj, prefix=""):
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_key = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_json(v, new_key))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            new_key = f"{prefix}[{i}]"
            out.update(flatten_json(v, new_key))
    else:
        out[prefix] = obj
    return out

def to_float_safe(x):
    try:
        if x is None:
            return None
        if isinstance(x, str):
            x_clean = x.replace("%", "").replace(",", "").strip()
            if x_clean.lower() in ["nan", "none", "null", ""]:
                return None
            return float(x_clean)
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None

def key_has_all_terms(key, terms):
    key = str(key).lower().replace("-", "_").replace(" ", "_")
    return all(str(term).lower().replace("-", "_").replace(" ", "_") in key for term in terms)

def find_metric(flat, split_terms, metric_terms, scope_term_groups=None, exclude_terms=None):
    exclude_terms = exclude_terms or []
    candidates = []

    for k, v in flat.items():
        key = str(k).lower()

        if any(term.lower() in key for term in exclude_terms):
            continue
        if split_terms and not key_has_all_terms(key, split_terms):
            continue
        if metric_terms and not key_has_all_terms(key, metric_terms):
            continue

        if scope_term_groups:
            scoped = any(key_has_all_terms(key, group) for group in scope_term_groups)
            if not scoped:
                continue

        val = to_float_safe(v)
        if val is not None and np.isfinite(val):
            candidates.append((k, val))

    if not candidates:
        return None, None

    candidates = sorted(candidates, key=lambda kv: len(kv[0]))
    return candidates[0][1], candidates[0][0]

def detect_best_candidate_name(flat, model_key):
    possible_keys = [
        "best_model", "selected_model", "best_candidate",
        "selected_candidate", "winning_model", "model_name", "candidate_name"
    ]

    for k, v in flat.items():
        key = str(k).lower()
        leaf = key.split(".")[-1]
        if leaf in possible_keys and isinstance(v, str) and len(v) < 160:
            scopes = MODEL_SCOPE_TERMS.get(model_key)
            if scopes and any(key_has_all_terms(key, group) for group in scopes):
                return v

    for k, v in flat.items():
        leaf = str(k).lower().split(".")[-1]
        if leaf in possible_keys and isinstance(v, str) and len(v) < 160:
            return v

    return None

def extract_metrics_from_artifact(model_key, artifact):
    flat = flatten_json(artifact)
    scope_terms = MODEL_SCOPE_TERMS.get(model_key, [])

    source_path = MODEL_ARTIFACTS.get(model_key)
    source_artifact = str(source_path.relative_to(REPO)) if source_path else "not found"

    row = {
        "model_key": model_key,
        "model_name": MODEL_DISPLAY_NAMES.get(model_key, model_key),
        "category": MODEL_CATEGORIES.get(model_key, "unknown"),
        "artifact_available": True,
        "source_artifact": source_artifact,
        "candidate_name": detect_best_candidate_name(flat, model_key),
        "validation_mae": None,
        "validation_mse": None,
        "validation_rmse": None,
        "validation_mape": None,
        "validation_bias": None,
        "validation_directional_accuracy": None,
        "test_mae": None,
        "test_mse": None,
        "test_rmse": None,
        "test_mape": None,
        "test_bias": None,
        "test_directional_accuracy": None,
        "metric_sources": {},
    }

    metric_aliases = {
        "mae": [["mae"], ["mean_absolute_error"]],
        "mse": [["mse"], ["mean_squared_error"]],
        "rmse": [["rmse"], ["root_mean_squared_error"]],
        "mape": [["mape"], ["mean_absolute_percentage_error"]],
        "bias": [["bias"], ["mean_error"]],
        "directional_accuracy": [["directional_accuracy"], ["direction_accuracy"], ["directional", "accuracy"]],
    }

    split_aliases_map = {
        "validation": [["validation"], ["valid"], ["val"]],
        "test": [["test"], ["holdout"], ["out_of_sample"], ["walk_forward"]],
    }

    # First pass: model-scoped metric extraction, useful for combined baseline artifacts.
    for split in ["validation", "test"]:
        for metric, alias_groups in metric_aliases.items():
            for split_alias in split_aliases_map[split]:
                for metric_alias in alias_groups:
                    found_val, found_path = find_metric(
                        flat,
                        split_alias,
                        metric_alias,
                        scope_term_groups=scope_terms,
                        exclude_terms=["train", "training", "in_sample", "aic", "bic"]
                    )
                    if found_val is not None:
                        row[f"{split}_{metric}"] = found_val
                        row["metric_sources"][f"{split}_{metric}"] = found_path
                        break
                if row[f"{split}_{metric}"] is not None:
                    break

    # Second pass: non-scoped extraction for dedicated per-model artifacts.
    for split in ["validation", "test"]:
        for metric, alias_groups in metric_aliases.items():
            if row[f"{split}_{metric}"] is not None:
                continue
            for split_alias in split_aliases_map[split]:
                for metric_alias in alias_groups:
                    found_val, found_path = find_metric(
                        flat,
                        split_alias,
                        metric_alias,
                        scope_term_groups=None,
                        exclude_terms=["train", "training", "in_sample", "aic", "bic"]
                    )
                    if found_val is not None:
                        row[f"{split}_{metric}"] = found_val
                        row["metric_sources"][f"{split}_{metric}"] = found_path
                        break
                if row[f"{split}_{metric}"] is not None:
                    break

    # Final fallback: use unsplit metrics only when no test metric exists.
    # This keeps old notebooks usable while avoiding train metrics.
    for metric, alias_groups in metric_aliases.items():
        if row[f"test_{metric}"] is None:
            for metric_alias in alias_groups:
                found_val, found_path = find_metric(
                    flat,
                    [],
                    metric_alias,
                    scope_term_groups=scope_terms,
                    exclude_terms=["train", "training", "in_sample", "aic", "bic"]
                )
                if found_val is None:
                    found_val, found_path = find_metric(
                        flat,
                        [],
                        metric_alias,
                        scope_term_groups=None,
                        exclude_terms=["train", "training", "in_sample", "aic", "bic"]
                    )
                if found_val is not None:
                    row[f"test_{metric}"] = found_val
                    row["metric_sources"][f"test_{metric}"] = found_path
                    break

    for col in ["validation_mape", "test_mape"]:
        if row[col] is not None and row[col] <= 1:
            row[col] = row[col] * 100.0

    for col in ["validation_directional_accuracy", "test_directional_accuracy"]:
        if row[col] is not None and row[col] <= 1:
            row[col] = row[col] * 100.0

    return row

comparison_rows = []
raw_model_artifacts = {}

for model_key, path in MODEL_ARTIFACTS.items():
    artifact = load_json(path) if path is not None else None
    if artifact is None:
        comparison_rows.append({
            "model_key": model_key,
            "model_name": MODEL_DISPLAY_NAMES.get(model_key, model_key),
            "category": MODEL_CATEGORIES.get(model_key, "unknown"),
            "artifact_available": False,
            "source_artifact": "not found",
            "candidate_name": None,
            "validation_mae": None,
            "validation_mse": None,
            "validation_rmse": None,
            "validation_mape": None,
            "validation_bias": None,
            "validation_directional_accuracy": None,
            "test_mae": None,
            "test_mse": None,
            "test_rmse": None,
            "test_mape": None,
            "test_bias": None,
            "test_directional_accuracy": None,
            "metric_sources": {},
        })
        continue

    raw_model_artifacts[model_key] = artifact
    comparison_rows.append(extract_metrics_from_artifact(model_key, artifact))

comparison_df = pd.DataFrame(comparison_rows)

comparison_df["primary_rmse"] = comparison_df["test_rmse"].combine_first(comparison_df["validation_rmse"])
comparison_df["primary_mae"] = comparison_df["test_mae"].combine_first(comparison_df["validation_mae"])
comparison_df["primary_mape"] = comparison_df["test_mape"].combine_first(comparison_df["validation_mape"])

comparison_df["ranking_basis"] = np.where(
    comparison_df["test_rmse"].notna(),
    "test_rmse_primary",
    np.where(comparison_df["validation_rmse"].notna(), "validation_rmse_fallback", "not_rankable")
)

rankable_df = comparison_df[
    (comparison_df["artifact_available"] == True) &
    (comparison_df["primary_rmse"].notna())
].copy()

rankable_df = rankable_df.sort_values(
    by=["primary_rmse", "primary_mae", "primary_mape"],
    ascending=[True, True, True],
    na_position="last"
).reset_index(drop=True)

rankable_df["rank"] = np.arange(1, len(rankable_df) + 1)

comparison_df = comparison_df.merge(rankable_df[["model_key", "rank"]], on="model_key", how="left")
comparison_df = comparison_df.sort_values(by=["rank", "model_key"], ascending=[True, True], na_position="last").reset_index(drop=True)

if len(rankable_df) == 0:
    display(comparison_df[["model_key", "artifact_available", "source_artifact", "ranking_basis"]])
    raise ValueError(
        "No rankable models found. At least one model needs validation/test RMSE in its JSON artifact. "
        "Check the model notebooks exported metrics under artifacts/models."
    )

selected_model_row = rankable_df.iloc[0].to_dict()

print("✅ Model comparison completed.")
print(f"Selected model by evidence: {selected_model_row['model_name']}")
print(f"Ranking basis: {selected_model_row['ranking_basis']}")
print(f"Primary RMSE: {selected_model_row['primary_rmse']:.4f}")

display_cols = [
    "rank", "model_key", "model_name", "category", "candidate_name",
    "validation_mae", "validation_rmse", "validation_mape",
    "test_mae", "test_rmse", "test_mape",
    "ranking_basis", "source_artifact"
]
display(comparison_df[display_cols])

## CELL 4B — Residual diagnostics from forecast path artifacts

In [ ]:
# =========================
# CELL 4B — RESIDUAL DIAGNOSTICS
# =========================

def find_first_col(columns, terms, exact_first=True):
    cols = list(columns)

    if exact_first:
        normalized = {str(c).lower().strip(): c for c in cols}
        for term in terms:
            term_norm = str(term).lower().strip()
            if term_norm in normalized:
                return normalized[term_norm]

    for col in cols:
        c = str(col).lower()
        if any(str(t).lower() in c for t in terms):
            return col
    return None

def normalize_forecast_records(obj):
    if obj is None:
        return []

    if isinstance(obj, list):
        if all(isinstance(x, dict) for x in obj):
            return obj
        return []

    if isinstance(obj, dict):
        priority_keys = [
            "forecast_path", "forecast_paths", "official_forecast", "chart_data",
            "data", "records", "test_forecast_path", "validation_forecast_path",
            "combined_forecast_path", "future_forecast", "forecast_records"
        ]
        for key in priority_keys:
            if key in obj:
                recs = normalize_forecast_records(obj[key])
                if recs:
                    return recs

        # Some artifacts have model-key dictionaries like {"naive": [...], "moving_average": [...]}.
        for key, value in obj.items():
            recs = normalize_forecast_records(value)
            if recs:
                return recs

        try:
            df = pd.DataFrame(obj)
            if len(df) > 0:
                return df.to_dict(orient="records")
        except Exception:
            pass

    return []

def load_forecast_path_as_df(path):
    path = Path(path)
    if not path.exists():
        return None

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    obj = load_json(path)
    recs = normalize_forecast_records(obj)
    if not recs:
        return None

    return pd.DataFrame(recs)

def filter_df_for_model(df, model_key):
    if df is None or df.empty:
        return df

    model_col = find_first_col(df.columns, ["model_key", "model", "method", "candidate", "forecast_type"])
    if model_col is None:
        return df

    temp = df.copy()
    model_terms = {
        "naive": ["naive"],
        "moving_average": ["moving", "average"],
        "exponential_smoothing": ["exponential", "smoothing", "ets"],
        "regression": ["regression"],
        "arima": ["arima"],
        "sarimax": ["sarimax"],
        "xgboost": ["xgboost", "xgb"],
        "prophet_optional": ["prophet"],
    }.get(model_key, [model_key])

    model_text = temp[model_col].astype(str).str.lower()
    mask = pd.Series(False, index=temp.index)
    for term in model_terms:
        mask = mask | model_text.str.contains(term, na=False)

    filtered = temp[mask].copy()
    return filtered if not filtered.empty else df

def compute_residual_summary(model_key, df):
    if df is None or df.empty:
        return None

    df = filter_df_for_model(df, model_key)

    date_col = find_first_col(df.columns, ["date", "ds"])
    actual_col = find_first_col(df.columns, ["actual", "actual_gold_price", "y_true", "observed", "gold_price", "y"])
    pred_col = find_first_col(df.columns, ["forecast", "prediction", "predicted", "yhat", "y_pred", "forecast_gold_price"])
    split_col = find_first_col(df.columns, ["split", "period", "dataset", "segment"])

    if actual_col is None or pred_col is None:
        return None

    temp = df.copy()
    temp[actual_col] = pd.to_numeric(temp[actual_col], errors="coerce")
    temp[pred_col] = pd.to_numeric(temp[pred_col], errors="coerce")
    temp = temp.dropna(subset=[actual_col, pred_col])

    if temp.empty:
        return None

    temp["residual"] = temp[actual_col] - temp[pred_col]
    temp["abs_error"] = temp["residual"].abs()
    temp["squared_error"] = temp["residual"] ** 2
    temp["ape"] = np.where(temp[actual_col] != 0, temp["abs_error"] / temp[actual_col].abs() * 100, np.nan)

    output = {
        "model_key": model_key,
        "model_name": MODEL_DISPLAY_NAMES.get(model_key, model_key),
        "records_used": int(len(temp)),
        "actual_column": str(actual_col),
        "forecast_column": str(pred_col),
        "mean_error_bias": float(temp["residual"].mean()),
        "mae_from_path": float(temp["abs_error"].mean()),
        "rmse_from_path": float(np.sqrt(temp["squared_error"].mean())),
        "mape_from_path": float(np.nanmean(temp["ape"])),
        "residual_std": float(temp["residual"].std()),
        "residual_min": float(temp["residual"].min()),
        "residual_max": float(temp["residual"].max()),
    }

    if date_col is not None:
        output["date_column"] = str(date_col)
        try:
            temp[date_col] = pd.to_datetime(temp[date_col], errors="coerce")
            if temp[date_col].notna().any():
                output["first_date"] = str(temp[date_col].min().date())
                output["last_date"] = str(temp[date_col].max().date())
        except Exception:
            pass

    if split_col is not None:
        output["split_column"] = str(split_col)
        output["splits_detected"] = [str(x) for x in sorted(temp[split_col].dropna().unique())]

    return output

residual_summaries = []
path_parse_notes = []

FORECAST_TO_MODEL_KEYS = {
    "naive_moving_average": ["naive", "moving_average"],
    "exponential_smoothing": ["exponential_smoothing"],
    "arima": ["arima"],
    "sarimax": ["sarimax"],
    "xgboost": ["xgboost"],
    "prophet_optional": ["prophet_optional"],
}

for path_key, path in FORECAST_PATHS.items():
    if path is None or not Path(path).exists():
        path_parse_notes.append({"path_key": path_key, "path": "not found", "status": "missing"})
        continue

    df_path = load_forecast_path_as_df(path)
    if df_path is None or df_path.empty:
        path_parse_notes.append({"path_key": path_key, "path": str(path.relative_to(REPO)), "status": "found_but_not_parsed"})
        continue

    model_keys = FORECAST_TO_MODEL_KEYS.get(path_key, [path_key])
    for model_key in model_keys:
        summary = compute_residual_summary(model_key, df_path)
        if summary:
            summary["source_path"] = str(path.relative_to(REPO))
            residual_summaries.append(summary)
            path_parse_notes.append({"path_key": path_key, "model_key": model_key, "path": str(path.relative_to(REPO)), "status": "parsed"})
        else:
            path_parse_notes.append({
                "path_key": path_key,
                "model_key": model_key,
                "path": str(path.relative_to(REPO)),
                "status": "could_not_identify_actual_forecast_columns"
            })

residual_df = pd.DataFrame(residual_summaries)

print("Residual diagnostics parsed:")
display(residual_df if not residual_df.empty else pd.DataFrame(columns=["model_key", "status"]))

print("Forecast path parsing notes:")
display(pd.DataFrame(path_parse_notes))

## CELL 5 — Artifact export

In [ ]:
# =========================
# CELL 5 — ARTIFACT EXPORT
# =========================

def clean_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, tuple):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [clean_for_json(v) for v in obj.tolist()]
    if isinstance(obj, (pd.Timestamp, datetime, date)):
        return obj.isoformat()
    if obj is pd.NaT:
        return None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clean_for_json(obj), f, indent=2, ensure_ascii=False)
    print(f"✅ Wrote {path.relative_to(REPO)}")

validation_by_model_records = comparison_df.drop(columns=["metric_sources"], errors="ignore").to_dict(orient="records")
validation_metric_sources = comparison_df[["model_key", "metric_sources"]].to_dict(orient="records")

ranking_records = rankable_df[[
    "rank", "model_key", "model_name", "category", "candidate_name",
    "primary_rmse", "primary_mae", "primary_mape",
    "validation_mae", "validation_rmse", "validation_mape",
    "test_mae", "test_rmse", "test_mape",
    "ranking_basis", "source_artifact"
]].to_dict(orient="records")

selected_model_summary = {
    "artifact_type": "selected_model_summary",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "selection_rule": {
        "primary_metric": "test_rmse_if_available_else_validation_rmse",
        "direction": "lower_is_better",
        "tie_breakers": ["mae", "mape"],
        "warning": "This notebook selects based on available JSON metrics. A model without exported RMSE cannot be ranked."
    },
    "selected_model": {
        "rank": int(selected_model_row["rank"]),
        "model_key": selected_model_row["model_key"],
        "model_name": selected_model_row["model_name"],
        "category": selected_model_row["category"],
        "candidate_name": selected_model_row.get("candidate_name"),
        "ranking_basis": selected_model_row["ranking_basis"],
        "primary_rmse": selected_model_row.get("primary_rmse"),
        "primary_mae": selected_model_row.get("primary_mae"),
        "primary_mape": selected_model_row.get("primary_mape"),
        "validation_metrics": {
            "mae": selected_model_row.get("validation_mae"),
            "rmse": selected_model_row.get("validation_rmse"),
            "mape": selected_model_row.get("validation_mape"),
            "bias": selected_model_row.get("validation_bias"),
            "directional_accuracy": selected_model_row.get("validation_directional_accuracy"),
        },
        "test_metrics": {
            "mae": selected_model_row.get("test_mae"),
            "rmse": selected_model_row.get("test_rmse"),
            "mape": selected_model_row.get("test_mape"),
            "bias": selected_model_row.get("test_bias"),
            "directional_accuracy": selected_model_row.get("test_directional_accuracy"),
        },
        "source_artifact": selected_model_row.get("source_artifact"),
    },
    "professor_safe_interpretation": [
        "The selected model is chosen by exported validation/test metrics, not by visual appearance or model complexity.",
        "Lower RMSE is the primary ranking criterion because it penalizes large forecast errors.",
        "Model comparison should be interpreted with the same cutoff date and the same test-period discipline.",
        "No model should be called final until this comparison artifact has been generated."
    ],
    "limitations": [
        "Metrics depend on the forecast mode exported by each model notebook.",
        "If one model uses static holdout and another uses walk-forward one-step forecasts, check forecast mode before making a final academic claim.",
        "Notebook 12 should use this selected_model_summary artifact rather than hardcoding a model winner."
    ],
}

validation_summary = {
    "artifact_type": "validation_summary",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "official_forecast_cutoff_date": OFFICIAL_CUTOFF_DATE,
    "models_expected": list(MODEL_ARTIFACTS.keys()),
    "models_available_count": int(comparison_df["artifact_available"].sum()),
    "models_expected_count": int(len(MODEL_ARTIFACTS)),
    "models_rankable_count": int(len(rankable_df)),
    "missing_model_artifacts": missing_artifacts,
    "missing_forecast_path_artifacts": missing_forecast_paths,
    "selected_model_key": selected_model_summary["selected_model"]["model_key"],
    "selected_model_name": selected_model_summary["selected_model"]["model_name"],
    "selected_model_primary_rmse": selected_model_summary["selected_model"]["primary_rmse"],
    "ranking_rule": selected_model_summary["selection_rule"],
    "professor_safe_summary": [
        "This page compares baseline, classical, explainable, economic time-series, nonlinear, and optional benchmark models.",
        "The ranking is evidence-based and generated from model artifacts.",
        "The website should read this JSON instead of hardcoding the winning model."
    ],
}

residual_diagnostics = {
    "artifact_type": "residual_diagnostics",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "residual_summaries": residual_summaries,
    "forecast_path_parse_notes": path_parse_notes,
    "limitations": [
        "Residual diagnostics are generated only when forecast path artifacts contain both actual and predicted values.",
        "If a forecast path could not be parsed, the corresponding model result metrics are still used for ranking if available."
    ]
}

page_model_comparison = {
    "artifact_type": "page_model_comparison",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "page_title": "Model Comparison and Validation",
    "page_subtitle": "Evidence-based comparison of Gold Nexus Alpha forecasting models.",
    "official_forecast_cutoff_date": OFFICIAL_CUTOFF_DATE,
    "selected_model": selected_model_summary["selected_model"],
    "kpi_cards": [
        {"label": "Models Expected", "value": int(len(MODEL_ARTIFACTS)), "description": "Forecasting methods listed in the project plan."},
        {"label": "Models Available", "value": int(comparison_df["artifact_available"].sum()), "description": "Model result artifacts found in the repository."},
        {"label": "Models Ranked", "value": int(len(rankable_df)), "description": "Models with usable RMSE metrics."},
        {"label": "Selected Model", "value": selected_model_summary["selected_model"]["model_name"], "description": "Selected by the ranking rule in this artifact."},
    ],
    "leaderboard": ranking_records,
    "validation_by_model": validation_by_model_records,
    "residual_diagnostics_available": len(residual_summaries) > 0,
    "charts": {
        "recommended_frontend_charts": [
            "RMSE comparison bar chart",
            "MAE comparison bar chart",
            "MAPE comparison bar chart",
            "Model ranking table",
            "Residual comparison chart if available"
        ]
    },
    "interpretation": [
        "This page is the professor-proof validation page.",
        "The model selected here is based on exported metrics, not visual hype.",
        "A model with better visual tracking still needs metric support before being declared the official model."
    ],
    "limitations": selected_model_summary["limitations"],
    "json_sources": [
        "artifacts/validation/validation_summary.json",
        "artifacts/validation/validation_by_model.json",
        "artifacts/validation/model_ranking.json",
        "artifacts/validation/residual_diagnostics.json",
        "artifacts/validation/selected_model_summary.json"
    ]
}

write_json(REPO / "artifacts/validation/validation_summary.json", validation_summary)
write_json(REPO / "artifacts/validation/validation_by_model.json", {
    "artifact_type": "validation_by_model",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "records": validation_by_model_records,
    "metric_sources": validation_metric_sources,
})
write_json(REPO / "artifacts/validation/model_ranking.json", {
    "artifact_type": "model_ranking",
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "ranking_rule": selected_model_summary["selection_rule"],
    "leaderboard": ranking_records,
})
write_json(REPO / "artifacts/validation/residual_diagnostics.json", residual_diagnostics)
write_json(REPO / "artifacts/validation/selected_model_summary.json", selected_model_summary)
write_json(REPO / "artifacts/pages/page_model_comparison.json", page_model_comparison)

comparison_csv_path = REPO / "artifacts/validation/validation_by_model.csv"
ranking_csv_path = REPO / "artifacts/validation/model_ranking.csv"

comparison_df.drop(columns=["metric_sources"], errors="ignore").to_csv(comparison_csv_path, index=False)
rankable_df.drop(columns=["metric_sources"], errors="ignore").to_csv(ranking_csv_path, index=False)

print(f"✅ Wrote {comparison_csv_path.relative_to(REPO)}")
print(f"✅ Wrote {ranking_csv_path.relative_to(REPO)}")

print("\nSelected model summary:")
print(json.dumps(clean_for_json(selected_model_summary["selected_model"]), indent=2))

## CELL 6 — GitHub push

In [ ]:
# =========================
# CELL 6 — GITHUB PUSH
# =========================

commit_message = "Add model comparison and validation artifacts"

run_cmd(["git", "config", "user.email", "colab@gold-nexus-alpha.local"], cwd=REPO)
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=REPO)

paths_to_add = [
    "artifacts/validation/validation_summary.json",
    "artifacts/validation/validation_by_model.json",
    "artifacts/validation/validation_by_model.csv",
    "artifacts/validation/model_ranking.json",
    "artifacts/validation/model_ranking.csv",
    "artifacts/validation/residual_diagnostics.json",
    "artifacts/validation/selected_model_summary.json",
    "artifacts/pages/page_model_comparison.json",
]

for p in paths_to_add:
    if (REPO / p).exists():
        run_cmd(["git", "add", p], cwd=REPO, check=False)
    else:
        print(f"Skipping missing file: {p}")

status = run_cmd(["git", "status", "--short"], cwd=REPO, check=False)

if not status.stdout.strip():
    print("No changes to commit.")
else:
    commit_result = run_cmd(["git", "commit", "-m", commit_message], cwd=REPO, check=False)

    if commit_result.returncode != 0:
        print("No commit created, possibly because files are unchanged.")
    else:
        pull_result = run_cmd(["git", "pull", "--rebase", "origin", BRANCH], cwd=REPO, check=False)
        if pull_result.returncode != 0:
            raise RuntimeError(
                "Git pull --rebase failed before push. Resolve conflicts, then run: git rebase --continue and git push origin main."
            )

        run_cmd(["git", "push", "origin", BRANCH], cwd=REPO)
        print("✅ Validation artifacts pushed to GitHub.")

# End of Notebook 11

After this notebook runs successfully, run:

`12_official_forecast_export.ipynb`

Notebook 12 will load:
- `artifacts/validation/selected_model_summary.json`
- `artifacts/validation/model_ranking.json`
- the selected model forecast path artifact

Then it exports the official forecast page JSON.